# Modélisation hybride ARIMAX + LSTM — prédiction du NO2

**Cible** : `NO2` par segment du périphérique (8 modèles indépendants).

**Approche** : modèle hybride en deux étages, conforme à la suggestion du cahier des
charges (*"modèles hybrides ARIMA + deep learning"*).

```
        ┌───────────┐                    ┌──────────────┐
NO2 ──▶ │  ARIMAX   │ ──▶ résidus  ──▶  │ LSTM (24h)   │ ──▶ correction
        └───────────┘                    └──────────────┘
              │                                                │
              └─────────── prédiction finale = ARIMAX + LSTM ──┘
```

- **ARIMAX** apprend la dynamique linéaire et l'effet des variables exogènes (météo,
  calendrier, validations FER).
- **LSTM** apprend les patterns non-linéaires que l'ARIMAX rate, à partir des **résidus**.

**Étapes** :
1. Chargement + features exogènes
2. Split temporel train (jan-août) / val (sept-oct) / test (nov-déc)
3. Vérification de stationnarité (justifie l'ordre ARIMA)
4. Pipeline ARIMAX + LSTM par segment
5. Évaluation comparée : baseline / ARIMAX seul / hybride
6. Comparaison globale avec XGBoost
7. Visualisation et sauvegarde

In [ ]:
import os, time, pickle, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

DATA_DIR = Path("../data/processed")
MODELS_DIR = Path("../src/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## Étape 1 — Chargement et configuration

**Variables exogènes choisies** : on garde les features journalières (météo + calendrier
+ validations FER) plus les features horaires cycliques. On **inclut `VALD_NAVIGO`**
explicitement — c'est notre variable d'hypothèse.

In [ ]:
df = pd.read_csv(DATA_DIR / "dataset_entrainement_2024.csv")
df["time"] = pd.to_datetime(df["time"])

EXOG_FEATURES = [
    # Horaires cycliques
    "HEURE_SIN", "HEURE_COS",
    # Calendrier
    "WEEKEND", "JOUR_FERIE", "VACANCES_SCOLAIRES", "JO",
    "JOUR_NON_OUVRE", "JOUR_PERTURBE",
    # Météo
    "TEMP_AVG_C", "WINDSPEED_MAX_KMH", "PRECIP_TOTAL_DAY_MM",
    "HUMIDITY_MAX_PERCENT", "PRESSURE_MAX_MB", "CLOUDCOVER_AVG_PERCENT",
    # Transport (notre variable d'hypothèse)
    "VALD_NAVIGO",
]

SEGMENTS = sorted(df["segment"].unique())
print(f"Dataset : {df.shape}")
print(f"Segments : {SEGMENTS}")
print(f"Features exogènes : {len(EXOG_FEATURES)}")

## Étape 2 — Split temporel

**Trois sets**, contrairement au notebook XGBoost qui n'en avait que deux. Le LSTM a
besoin d'un set de validation **séparé** pour l'early stopping, sinon on tomberait
dans le data leakage.

| Set | Période | Usage |
|-----|---------|-------|
| **Train** | janv → août 2024 | Entraîne ARIMAX et LSTM |
| **Validation** | sept → oct 2024 | Early stopping du LSTM |
| **Test** | nov → déc 2024 | Évaluation finale, jamais vu pendant l'entraînement |

In [ ]:
def split_temporal(data):
    train = data[data["time"] <  "2024-09-01"].copy()
    val   = data[(data["time"] >= "2024-09-01") & (data["time"] < "2024-11-01")].copy()
    test  = data[data["time"] >= "2024-11-01"].copy()
    return train, val, test

# Vérification sur un segment
seg_demo = df[df["segment"] == SEGMENTS[0]].sort_values("time").reset_index(drop=True)
tr, va, te = split_temporal(seg_demo)
print(f"Train : {len(tr):>5} lignes  ({tr['time'].min().date()} → {tr['time'].max().date()})")
print(f"Val   : {len(va):>5} lignes  ({va['time'].min().date()} → {va['time'].max().date()})")
print(f"Test  : {len(te):>5} lignes  ({te['time'].min().date()} → {te['time'].max().date()})")

## Étape 3 — Vérification de stationnarité (test ADF)

ARIMA(p,**d**,q) requiert de choisir `d` (nombre de différenciations pour rendre la
série stationnaire). On utilise le **test Augmented Dickey-Fuller** :

- p-value < 0.05 → série déjà stationnaire → **d = 0**
- p-value ≥ 0.05 → différencier une fois → d = 1

C'est plus rigoureux que de choisir d arbitrairement.

In [ ]:
print("Test ADF de stationnarité du NO2 par segment (train set) :\n")
for seg in SEGMENTS:
    train, _, _ = split_temporal(df[df["segment"] == seg].sort_values("time"))
    pval = adfuller(train["NO2"].dropna())[1]
    statut = "stationnaire ✓" if pval < 0.05 else "non stationnaire"
    print(f"  {seg:15}  p-value = {pval:.6f}  → {statut}")

print("\n→ Toutes les séries sont stationnaires : on utilisera ARIMAX(2, 0, 1)")

## Étape 4 — Pipeline ARIMAX + LSTM

Fonction qui pour un segment donné :
1. Split + standardisation des exogènes (fit sur train uniquement)
2. ARIMAX sur train → prédictions val + test
3. Calcul des **résidus** = NO2 réel − ARIMAX prédit
4. LSTM entraîné sur résidus **train**, avec early stopping sur val
5. Prédiction hybride = ARIMAX + LSTM(résidus) sur le test
6. Métriques baseline / ARIMAX / hybride

**Points critiques (corrige les bugs du script initial)** :
- Le LSTM voit les **résidus du train** pour s'entraîner, pas ceux du test
- L'early stopping utilise les résidus **val**
- Les métriques sont calculées sur le test, jamais vu

In [ ]:
LOOK_BACK = 24   # 24h d'historique pour le LSTM

def make_sequences(arr, lb):
    """Transforme une série en (X séquences, y cibles)."""
    X, y = [], []
    for i in range(len(arr) - lb):
        X.append(arr[i:i+lb])
        y.append(arr[i+lb])
    return np.array(X).reshape(-1, lb, 1), np.array(y)


def pipeline_segment(data, exog_cols, look_back=24, save_prefix=None):
    """Pipeline complet ARIMAX+LSTM pour un segment.

    Renvoie (dict de métriques, dict de prédictions test).
    """
    train, val, test = split_temporal(data)

    # 1. Standardisation des exogènes (fit sur train SEULEMENT)
    scaler_exog = StandardScaler()
    exog_tr = pd.DataFrame(scaler_exog.fit_transform(train[exog_cols]),
                           columns=exog_cols, index=train.index)
    exog_va = pd.DataFrame(scaler_exog.transform(val[exog_cols]),
                           columns=exog_cols, index=val.index)
    exog_te = pd.DataFrame(scaler_exog.transform(test[exog_cols]),
                           columns=exog_cols, index=test.index)

    # 2. ARIMAX
    arimax = ARIMA(train["NO2"], exog=exog_tr, order=(2, 0, 1)).fit()
    pred_tr_arimax = arimax.fittedvalues
    pred_va_arimax = arimax.forecast(steps=len(val),  exog=exog_va)
    pred_te_arimax = arimax.forecast(steps=len(test), exog=exog_te)

    # 3. Résidus
    res_tr = train["NO2"].values - pred_tr_arimax.values
    res_va = val["NO2"].values   - pred_va_arimax.values
    res_te = test["NO2"].values  - pred_te_arimax.values

    # 4. LSTM sur résidus (scaler fit sur train uniquement)
    scaler_res = MinMaxScaler()
    res_tr_s = scaler_res.fit_transform(res_tr.reshape(-1,1)).flatten()
    res_va_s = scaler_res.transform(res_va.reshape(-1,1)).flatten()
    res_te_s = scaler_res.transform(res_te.reshape(-1,1)).flatten()

    Xtr, ytr = make_sequences(res_tr_s, look_back)
    Xva, yva = make_sequences(res_va_s, look_back)
    Xte, yte = make_sequences(res_te_s, look_back)

    lstm = Sequential([
        LSTM(50, return_sequences=True, input_shape=(look_back, 1)),
        Dropout(0.2),
        LSTM(50),
        Dense(1),
    ])
    lstm.compile(optimizer="adam", loss="mse")
    es = EarlyStopping(patience=3, restore_best_weights=True)
    lstm.fit(Xtr, ytr, validation_data=(Xva, yva),
             epochs=20, batch_size=32, callbacks=[es], verbose=0)

    # 5. Prédiction hybride sur test
    pred_res_te_s = lstm.predict(Xte, verbose=0).flatten()
    pred_res_te   = scaler_res.inverse_transform(pred_res_te_s.reshape(-1,1)).flatten()

    # Alignement temporel : on perd les LOOK_BACK premières heures du test
    y_te_aligned     = test["NO2"].values[look_back:]
    arimax_aligned   = pred_te_arimax.values[look_back:]
    hybrid_aligned   = arimax_aligned + pred_res_te
    test_aligned_idx = test.index[look_back:]

    # 6. Baseline : moyenne train par (heure × jour-semaine)
    train["__h"] = train["time"].dt.hour
    train["__d"] = train["time"].dt.dayofweek
    baseline_table = train.groupby(["__h", "__d"])["NO2"].mean()
    test_h = test["time"].dt.hour.values[look_back:]
    test_d = test["time"].dt.dayofweek.values[look_back:]
    baseline_pred = np.array([baseline_table.get((h,d), train["NO2"].mean())
                              for h, d in zip(test_h, test_d)])

    # Métriques
    def met(y, yp):
        return {
            "RMSE": np.sqrt(mean_squared_error(y, yp)),
            "MAE":  mean_absolute_error(y, yp),
            "R²":   r2_score(y, yp),
        }

    metrics = {
        "Baseline":    met(y_te_aligned, baseline_pred),
        "ARIMAX":      met(y_te_aligned, arimax_aligned),
        "ARIMAX+LSTM": met(y_te_aligned, hybrid_aligned),
    }

    # Sauvegarde (optionnelle)
    if save_prefix:
        with open(MODELS_DIR / f"{save_prefix}_arimax.pkl", "wb") as f:
            pickle.dump(arimax, f)
        lstm.save(MODELS_DIR / f"{save_prefix}_lstm.keras")
        joblib.dump(scaler_exog, MODELS_DIR / f"{save_prefix}_scaler_exog.joblib")
        joblib.dump(scaler_res,  MODELS_DIR / f"{save_prefix}_scaler_res.joblib")

    predictions = {
        "y_true":      y_te_aligned,
        "baseline":    baseline_pred,
        "arimax":      arimax_aligned,
        "hybrid":      hybrid_aligned,
        "time":        test.iloc[look_back:]["time"].values,
    }
    return metrics, predictions

print("Fonction pipeline_segment définie.")

## Entraînement sur les 8 segments

Compte environ **~20 secondes par segment**, soit ~3 minutes au total.

In [ ]:
results = {}
predictions = {}

t0_global = time.time()
for seg in SEGMENTS:
    print(f"📍 {seg} ...", end=" ", flush=True)
    t0 = time.time()
    data_seg = df[df["segment"] == seg].sort_values("time").reset_index(drop=True)
    save_prefix = f"hybrid_NO2_{seg.replace('-', '_')}"
    metrics_seg, preds_seg = pipeline_segment(
        data_seg, EXOG_FEATURES,
        look_back=LOOK_BACK,
        save_prefix=save_prefix,
    )
    results[seg] = metrics_seg
    predictions[seg] = preds_seg
    print(f"({time.time()-t0:.0f}s)")

print(f"\n✓ Tous segments entraînés en {time.time()-t0_global:.0f}s")

## Étape 5 — Évaluation comparée par segment

In [ ]:
rows = []
for seg, metrics_seg in results.items():
    for model_name, m in metrics_seg.items():
        rows.append({
            "Segment": seg, "Modèle": model_name,
            "RMSE": round(m["RMSE"], 2),
            "MAE":  round(m["MAE"],  2),
            "R²":   round(m["R²"],   3),
        })

df_results = pd.DataFrame(rows)
print(df_results.pivot(index="Segment", columns="Modèle", values="RMSE").round(2))
df_results

In [ ]:
# Moyenne sur les 8 segments
print("Performance moyenne (sur les 8 segments) :\n")
moyennes = df_results.groupby("Modèle")[["RMSE", "MAE", "R²"]].mean().round(3)
print(moyennes.to_string())

# Gain hybride vs baseline et vs ARIMAX seul
gain_base   = (moyennes.loc["Baseline","RMSE"]    - moyennes.loc["ARIMAX+LSTM","RMSE"]) / moyennes.loc["Baseline","RMSE"]    * 100
gain_arimax = (moyennes.loc["ARIMAX","RMSE"]      - moyennes.loc["ARIMAX+LSTM","RMSE"]) / moyennes.loc["ARIMAX","RMSE"]      * 100
print(f"\nGain RMSE hybride vs baseline : -{gain_base:.1f} %")
print(f"Gain RMSE hybride vs ARIMAX seul : -{gain_arimax:.1f} %")

## Étape 6 — Comparaison avec XGBoost

XGBoost a été entraîné sur **les 8 segments simultanément** (un seul modèle), alors
que l'hybride est un **modèle par segment**. La comparaison reste légitime mais cette
différence d'architecture est à mentionner dans le rapport.

In [ ]:
# On rapproche les chiffres XGBoost (du notebook précédent, set de test nov-déc)
# Si le modèle XGBoost a été sauvegardé, on peut le réévaluer ; sinon on cite les résultats.
xgb_path = MODELS_DIR / "xgboost_no2.json"
if xgb_path.exists():
    from xgboost import XGBRegressor
    print("Chargement et évaluation de XGBoost...")
    # On recharge le dataset comme dans le notebook XGBoost
    df_xgb = pd.read_csv(DATA_DIR / "dataset_entrainement_2024.csv")
    df_xgb = df_xgb.drop(columns=["time", "NOM_JOUR", "NOM_FERIE", "PM10", "PM25"])
    df_xgb["TYPE_VACANCES"] = df_xgb["TYPE_VACANCES"].fillna("Aucune")
    df_xgb = pd.get_dummies(df_xgb, columns=["segment", "PERIODE", "TYPE_VACANCES"], dtype=int)
    df_xgb["DATE"] = pd.to_datetime(df_xgb["DATE"])
    test_xgb = df_xgb[df_xgb["DATE"] >= "2024-11-01"].copy()
    features_path = MODELS_DIR / "xgboost_no2_features.txt"
    features = features_path.read_text().split("\n")
    # S'assurer que toutes les features sont là (catégorie absente du test → 0)
    for f in features:
        if f not in test_xgb.columns:
            test_xgb[f] = 0
    model_xgb = XGBRegressor()
    model_xgb.load_model(xgb_path)
    y_test_xgb = test_xgb["NO2"]
    y_pred_xgb = model_xgb.predict(test_xgb[features])
    rmse_xgb = np.sqrt(mean_squared_error(y_test_xgb, y_pred_xgb))
    mae_xgb  = mean_absolute_error(y_test_xgb, y_pred_xgb)
    r2_xgb   = r2_score(y_test_xgb, y_pred_xgb)
    print(f"XGBoost (tous segments) : RMSE={rmse_xgb:.2f}  MAE={mae_xgb:.2f}  R²={r2_xgb:.3f}")
else:
    print("⚠ Modèle XGBoost non trouvé — exécute d'abord modelisation_xgboost.ipynb")

# Tableau récap final
print(f"\n--- Récapitulatif global (test set nov-déc 2024) ---")
print(moyennes.to_string())
if xgb_path.exists():
    print(f"  XGBoost             RMSE={rmse_xgb:.2f}  MAE={mae_xgb:.2f}  R²={r2_xgb:.3f}")

## Étape 7 — Visualisation des prédictions

Première semaine de novembre sur un segment, pour voir la qualité de reconstruction
horaire.

In [ ]:
seg_viz = "Chap-Bagn"
preds = predictions[seg_viz]

# Sélection première semaine du test
times = pd.to_datetime(preds["time"])
mask = times < "2024-11-08"

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(times[mask], preds["y_true"][mask],   label="Observé",      linewidth=1.5, color="black")
ax.plot(times[mask], preds["arimax"][mask],   label="ARIMAX seul",  linewidth=1.2, alpha=0.7)
ax.plot(times[mask], preds["hybrid"][mask],   label="ARIMAX + LSTM",linewidth=1.2, alpha=0.9)
ax.set_xlabel("Temps")
ax.set_ylabel("NO2 (µg/m³)")
ax.set_title(f"Prédictions sur la 1ère semaine du test — segment {seg_viz}")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison RMSE par segment sous forme de barres
fig, ax = plt.subplots(figsize=(11, 5))
piv = df_results.pivot(index="Segment", columns="Modèle", values="RMSE")
piv = piv[["Baseline", "ARIMAX", "ARIMAX+LSTM"]]
piv.plot(kind="bar", ax=ax, color=["#999999", "#1f77b4", "#d62728"])
ax.set_ylabel("RMSE (µg/m³)")
ax.set_title("RMSE par segment et par modèle (test nov-déc 2024)")
ax.grid(alpha=0.3, axis="y")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

---

## Récap pour le mémoire

**Modèles comparés** : Baseline (moyenne historique), ARIMAX, ARIMAX+LSTM (hybride),
XGBoost — 4 approches au total.

**Architecture hybride** : ARIMAX capte la dynamique linéaire + l'effet des exogènes
(météo, calendrier, validations) ; LSTM apprend les patterns non-linéaires que l'ARIMAX
rate, à partir des résidus.

**Validation du choix méthodologique** : test ADF justifie d=0 dans ARIMA. Split
train/val/test temporel évite le data leakage. Early stopping sur val pour le LSTM.

**Conformité au cahier des charges** :
- *"modèles hybrides ARIMA + deep learning"* → ✓ ARIMAX + LSTM
- *"modèles prédictifs mis en œuvre"* (pluriel) → ✓ 4 modèles comparés
- *"métriques de performance"* → ✓ RMSE / MAE / R² par segment et global